# Spectral Joint Flow (SJFPO) — Online Colab / Kaggle (with internet)

Clone the repo with a GitHub token, install requirements, then run training
via `!python scripts/train.py`. No source is embedded in the notebook.

## 0. Secrets and environment

1. Set `GITHUB_TOKEN` in the environment (Colab secrets or Kaggle secrets).
2. The token is read via `load_secret()` using the runtime environment.
3. Run all cells top to bottom.

In [ ]:
# Detect runtime and load secrets from the right store.
import os

OUTPUT_DIR = {
    "colab": "/content/",
    "kaggle": "/kaggle/working/",
    "local": "./",
}
INPUT_DIR = {
    "colab": "/content/",
    "kaggle": "/kaggle/input/",
    "local": "./",
}


def detect_env():
    try:
        if "google.colab" in str(get_ipython()):
            return "colab"
    except NameError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    return "local"


ENV_NAME = detect_env()
print(f"Environment: {ENV_NAME}")


def load_secret(key_name):
    try:
        if ENV_NAME == "colab":
            from google.colab import userdata

            return userdata.get(key_name) or None
        if ENV_NAME == "kaggle":
            from kaggle_secrets import UserSecretsClient

            return UserSecretsClient().get_secret(key_name) or None
        return os.getenv(key_name)
    except Exception as e:
        print(f"load_secret({key_name}) failed: {e}")
        return None


ENV_NAME = detect_env()
OUTPUT = OUTPUT_DIR[ENV_NAME]

Environment: kaggle


In [ ]:
# Clone the repo using the GitHub token from the environment/store.
%cd $OUTPUT
!rm -rf spectral-joint-policy
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")
repo_slug = os.getenv("GITHUB_REPO", "dzungphieuluuky/spectral-joint-policy")
repo_url = f"https://github.com/{repo_slug}.git"
if GITHUB_TOKEN:
    repo_url = repo_url.replace("github.com", f"{GITHUB_TOKEN}@github.com")
print(
    f"Cloning from: {repo_url.replace(GITHUB_TOKEN or '', '<TOKEN>') if GITHUB_TOKEN else repo_url}"
)
!git clone $repo_url spectral-joint-policy
# Move into the cloned repo and run directly (no embedded source).
%cd spectral-joint-policy
!pwd

/kaggle/working
Cloning from: https://<TOKEN>@github.com/dzungphieuluuky/spectral-joint-policy.git
Cloning into 'spectral-joint-policy'...
remote: Enumerating objects: 2604, done.
remote: Counting objects: 100% (481/481), done.
remote: Compressing objects: 100% (197/197), done.
remote: Total 2604 (delta 281), reused 471 (delta 280), pack-reused 2123 (from 1)
Receiving objects: 100% (2604/2604), 1.60 MiB | 4.67 MiB/s, done.
Resolving deltas: 100% (1780/1780), done.
/kaggle/working/spectral-joint-policy
/kaggle/working/spectral-joint-policy


In [ ]:
%%writefile requirements-colab.txt
# Deep Learning
torch
torchvision
torchaudio
torchcodec
jax
jaxlib
accelerate
diffusers
datasets
huggingface-hub
safetensors
bitsandbytes
einops
einx

# Reinforcement Learning / Robot Learning
gymnasium
gymnasium-robotics
pettingzoo
mujoco
mujoco-mjx
mujoco-warp
skrl
lerobot
jax[cuda12]
flax
jaxlib
optax
orbax-checkpoint

# Scientific / Vision
numpy
scipy
pandas
matplotlib
scikit-learn
scikit-image
opencv-python-headless
pillow
imageio
imageio-ffmpeg
av
moviepy

# Data / Storage
pyarrow

# Experiment Tracking / Config
wandb
tensorboard
draccus
tqdm

Overwriting requirements-colab.txt


In [ ]:
# Detect runtime and choose install source.
if ENV_NAME == "colab":
    !uv pip install -r requirements-colab.txt
    # !sudo apt-get update
    # !sudo apt-get install -y pkg-config python3-dev \
    # libavformat-dev libavcodec-dev libavdevice-dev \
    # libavutil-dev libavfilter-dev libswscale-dev libswresample-dev
    !uv pip uninstall av
    !uv pip install "av<15.0.0" --only-binary=:all:
else:
    !uv pip install -r requirements-colab.txt
    # !sudo apt-get update
    # !sudo apt-get install -y pkg-config python3-dev \
    # libavformat-dev libavcodec-dev libavdevice-dev \
    # libavutil-dev libavfilter-dev libswscale-dev libswresample-dev
    !uv pip uninstall av
    !uv pip install "av<15.0.0" --only-binary=:all:

Using Python 3.12.13 environment at: /usr
Resolved 160 packages in 1.54s
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
decorator            ------------------------------     0 B/9.02 KiB
⠙ Preparing packages... (0/16)
decorator            ------------------------------ 9.02 KiB/9.02 KiB
⠙ Preparing packages... (0/16)
decorator            ------------------------------ 9.02 KiB/9.02 KiB
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
draccus              ------------------------------ 14.81 KiB/83.45 KiB
⠙ Preparing packages... (0/16)
draccus              ------------------------------ 30.81 KiB/83.45 KiB
⠙ Preparing packages... (0/16)
draccus              ------------------------------ 46.81 KiB/83.45 KiB
⠙ Preparing packages... (0/16)
draccus              ------------------------------ 62.81 KiB/83.45 KiB
⠙ Preparing packages... (0/16)
draccus              ------

In [ ]:
# wandb login from secret store (runs before any training cell).
_wandb_key = load_secret("WANDB_API_KEY")
ifwandb_key:
    !wandb login $_wandb_key
else:
    print("WANDB_API_KEY not found; training will run without wandb sync.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [ ]:
# Run training directly from the cloned repo.
!SJFPO_COMPILE=0 SJFPO_DISABLE_WANDB=0 SJFPO_MJX=0 MPLBACKEND=Agg MUJOCO_GL=egl python scripts/train.py \
    --method sjfpo \
    --env Hopper-v5 \
    --seed 3407 \
    --total_steps 1000000 \
    --eval_steps 100 \
    --save_steps 1000 \
    --sjfpo.num_envs 64 \
    --sjfpo.chunks_per_update 256 \
    --sjfpo.num_minibatches 4 \
    --sjfpo.use_per_bin_clip false \
    --sjfpo.use_clipped_value_loss true \
    --sjfpo.use_dct_transform true

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
SJFPOPolicy(
  (actor): SJFPOFlowModel(
    (timestep_embedder): SinusoidalEncoding()
    (in_proj): Linear(in_features=17, out_features=128, bias=True)
    (bin_encoding): SinusoidalEncoding()
    (time_mlp): CondMLP(
      (fc1): Linear(in_features=128, out_features=512, bias=True)
      (act): SiLU()
      (fc2): Linear(in_features=512, out_features=128, bias=True)
    )
    (blocks): ModuleList(
      (0-3): 4 x DiTBlock(
        (norm1): RMSNorm((128,), eps=1e-06, elementwise_affine=True)
        (attn): SelfAttention(
          (qkv): Linear(in_features=128, out_features=384, bias=True)
          (out): Linear(in_features=128, out_features=128, bias=True)
        )
        (norm2): RMSNo